In [1]:
"""
get_mask_hidrica.py
===================
Módulo de geração de máscara hídrica a partir da banda NIR (M6C03) do
satélite GOES-16, com o objetivo de remover corpos d'água da análise de
índices espectrais.

Princípio físico:
    Corpos d'água absorvem fortemente a radiação no infravermelho próximo (NIR).
    Pixels com reflectância NIR abaixo de um limiar (padrão: 0.08) são
    classificados como água ou superfície não-vegetada e recebem valor 0.
    Pixels acima do limiar recebem valor 1 (vegetação/solo válido).

Fluxo principal:
    1. Filtragem dos arquivos GeoTIFF da banda NIR (padrão: contém 'M6C03').
    2. Leitura dos dados de reflectância.
    3. Preservação de pixels NaN (sem dado) — não classificados como água.
    4. Limiarização: reflectância < threshold → 0 (água); ≥ threshold → 1 (válido).
    5. Exportação como GeoTIFF uint8 com compressão LZW.

Dependências:
    rasterio, numpy

Uso típico:
    batch_reclassify(
        input_dir='CUT',
        output_dir='MASCARA_HIDRICA',
    )
"""

import os
from typing import List

import numpy as np
import rasterio

# Fallback para tqdm: se não instalado, usa iterador simples com print
try:
    from tqdm import tqdm
except ImportError:
    def tqdm(iterable, desc="", **kwargs):
        """Substituto simples de tqdm quando a biblioteca não está instalada."""
        print(f"{desc}..." if desc else "Processando...")
        return iterable

# ---------------------------------------------------------------------------
# Constantes do módulo
# ---------------------------------------------------------------------------

# Limiar de reflectância NIR para separação água/vegetação.
# Pixels abaixo deste valor são classificados como água ou superfície não-vegetada.
# Referência: valor empírico amplamente utilizado em sensoriamento remoto.
NIR_WATER_THRESHOLD: float = 0.08

# Substring identificadora da banda NIR nos nomes de arquivo GOES-16
NIR_BAND_ID: str = "M6C03"

# Prefixo adicionado ao nome dos arquivos de saída
OUTPUT_PREFIX: str = "reclass_"

# Valores da máscara binária de saída
MASK_WATER:   np.uint8 = np.uint8(0)   # água / superfície não-vegetada
MASK_VALID:   np.uint8 = np.uint8(1)   # vegetação / solo válido
MASK_NODATA:  np.uint8 = np.uint8(255) # sem dado (NaN na entrada)


# ---------------------------------------------------------------------------
# Funções auxiliares (uso interno)
# ---------------------------------------------------------------------------

def _reclassify_nir(
    data: np.ndarray,
    threshold: float,
) -> np.ndarray:
    """
    Reclassifica um array de reflectância NIR em máscara binária uint8.

    Regras de classificação:
        - NaN na entrada           → MASK_NODATA (255) — preserva ausência de dado
        - reflectância < threshold → MASK_WATER  (0)   — água / não-vegetado
        - reflectância ≥ threshold → MASK_VALID  (1)   — vegetação / solo válido

    Parâmetros:
        data      (np.ndarray): Array 2D de reflectância NIR (float32).
        threshold (float):      Limiar de separação água/vegetação.

    Retorna:
        np.ndarray: Máscara binária dtype uint8.
    """
    result = np.where(data >= threshold, MASK_VALID, MASK_WATER).astype(np.uint8)

    # Preserva pixels sem dado: NaN na entrada → 255 na saída
    nan_mask = np.isnan(data)
    result[nan_mask] = MASK_NODATA

    return result


# ---------------------------------------------------------------------------
# Função pública
# ---------------------------------------------------------------------------

def batch_reclassify(
    input_dir: str,
    output_dir: str,
    threshold: float = NIR_WATER_THRESHOLD,
    band_id: str = NIR_BAND_ID,
    output_prefix: str = OUTPUT_PREFIX,
) -> int:
    """
    Gera máscaras hídricas em lote a partir de arquivos GeoTIFF da banda NIR.

    Processa todos os arquivos `.tif` cujo nome contenha `band_id` (padrão:
    'M6C03'), aplicando limiarização NIR para separar água de vegetação/solo.

    Parâmetros:
        input_dir     (str):   Diretório contendo os GeoTIFFs de entrada.
        output_dir    (str):   Diretório de saída para as máscaras geradas.
        threshold     (float): Limiar NIR para classificação (padrão: 0.08).
        band_id       (str):   Substring identificadora da banda NIR no nome
                               do arquivo (padrão: 'M6C03', case-insensitive).
        output_prefix (str):   Prefixo dos arquivos de saída (padrão: 'reclass_').

    Retorna:
        int: Número de arquivos processados com sucesso.

    Levanta:
        FileNotFoundError: Se `input_dir` não existir.
    """
    if not os.path.isdir(input_dir):
        raise FileNotFoundError(f"Diretório de entrada não encontrado: '{input_dir}'")

    # Filtragem case-insensitive pela banda NIR
    tif_files = sorted(
        f for f in os.listdir(input_dir)
        if f.lower().endswith(".tif") and band_id.upper() in f.upper()
    )

    if not tif_files:
        print(f"⚠️  Nenhum arquivo .tif com '{band_id}' encontrado em '{input_dir}'.")
        return 0

    os.makedirs(output_dir, exist_ok=True)

    print(f"📁 Entrada  : {input_dir}")
    print(f"📁 Saída    : {output_dir}")
    print(f"📊 Arquivos : {len(tif_files)}")
    print(f"🌊 Limiar NIR: {threshold}\n")

    success = 0
    errors: List[str] = []

    for file in tqdm(tif_files, desc="Gerando máscaras hídricas"):
        input_path  = os.path.join(input_dir, file)
        output_path = os.path.join(output_dir, f"{output_prefix}{file}")

        try:
            with rasterio.open(input_path) as src:
                data = src.read(1).astype(np.float32)
                meta = src.meta.copy()

            # Reclassifica: NaN preservado, < threshold → 0, ≥ threshold → 1
            mask = _reclassify_nir(data, threshold)

            # Máscara binária uint8: menor dtype possível para 3 valores (0, 1, 255)
            meta.update({
                "dtype":   "uint8",
                "nodata":  int(MASK_NODATA),  # 255 — distinguível dos valores válidos
                "compress": "lzw",
                "tiled":   True,
            })

            with rasterio.open(output_path, "w", **meta) as dst:
                dst.write(mask, 1)

            success += 1

        except Exception as e:
            errors.append(file)
            print(f"\n  ✘ Erro em '{file}': {e}")
            continue

    # Resumo final
    print(f"\n{'=' * 50}")
    print(f"✅ Sucesso : {success}/{len(tif_files)}")
    if errors:
        print(f"❌ Erros   : {len(errors)}/{len(tif_files)}")
        for name in errors:
            print(f"   • {name}")
    print(f"💾 Resultados em: {output_dir}")
    print("=" * 50)

    return success


# ---------------------------------------------------------------------------
# Exemplo de uso
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    batch_reclassify(
        input_dir="CUT",
        output_dir="MASCARA_HIDRICA",
    )

📁 Entrada  : CUT
📁 Saída    : MASCARA_HIDRICA
📊 Arquivos : 12
🌊 Limiar NIR: 0.08



Gerando máscaras hídricas: 100%|██████████| 12/12 [00:00<00:00, 64.63it/s]


✅ Sucesso : 12/12
💾 Resultados em: MASCARA_HIDRICA
